<a href="https://course.bioinfo.se/open.html?nb=session08-lab.ipynb" style="display:inline-block;padding:10px 18px;background-color:#007ACC;color:#FFFFFF;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in VS Code</a>

# Session 8 lab: data, models and evaluation on blood cells {.unnumbered}

This lab is the hands-on half of Session 8. It uses **BloodMNIST**:
light-microscope images of single blood cells from healthy donors, in
eight classes, shrunk to 28×28 colour pixels. Recognizing white blood
cell types is a real clinical task (a "differential count" is part of
every routine blood test, and automated microscopes already do it), and
the cell types are related to each other in ways that show up in the
model's mistakes. At the end the lab comes back to proteins for the
homology question.

Every `todo("...")` call marks a piece of code for you to write: replace
the whole `todo(...)` call with your code. Work through the cells **in
order, top to bottom**. Until you fill it in, a cell stops with
`NotImplementedError: TODO in this cell: ...`, which tells you what is
missing. That is expected.

Session 8 does not teach *training* yet. The notebook gives you a ready-made
`train_model(...)` function; Session 9 opens that box (backpropagation, the
training loop, cross-entropy). Here you build the data sets and the
model, and you **evaluate** what comes out.

**Runtime:** a few minutes on Colab's free CPU. Keep `SEED = 0` so that
your numbers match the lab quiz (neural-network training still varies a
little between machines, so the quiz accepts a small range).

*Data:* BloodMNIST (MedMNIST v2; Yang et al. 2023, *Sci Data* 10:41;
CC BY 4.0), resized from the blood-cell images of Acevedo et al. 2020,
*Data in Brief* 30:105474.

In [ ]:
import os, re, tempfile, urllib.request
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score, matthews_corrcoef
from sklearn.model_selection import train_test_split

SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.set_num_threads(max(1, os.cpu_count() or 1))

def todo(what):
    """Placeholder for code you write: replace the whole todo(...) call with your own code."""
    raise NotImplementedError(f"TODO in this cell: {what}")

## 1. The blood cells and their classes

The full BloodMNIST has 17,092 images. This lab uses a fixed random
subset of its official training, validation and test splits (4,000,
1,000 and 2,000 images), small enough to download quickly and train on a
CPU. The classes keep their natural proportions. (Set `USE_FULL_DATA =
True` to use all 17,092 images instead, a 35 MB download; the quiz
numbers refer to the subset.)

The eight classes:

| Class | What it is |
|---|---|
| basophil | rare white blood cell (under 1% of white cells in blood); large dark-purple granules, involved in allergy |
| eosinophil | white blood cell with large orange-red granules; fights parasites, involved in allergy |
| erythroblast | an immature red blood cell that still has its nucleus; normally stays in the bone marrow |
| immature granulocyte | precursors (promyelocytes, myelocytes, metamyelocytes) of the granulocytes, mostly of neutrophils; normally in the bone marrow |
| lymphocyte | small white blood cell with a round nucleus that fills most of the cell; B and T cells |
| monocyte | the largest white blood cell, with a folded, kidney-shaped nucleus; becomes a macrophage in tissue |
| neutrophil | the most common white blood cell; nucleus in 3-5 lobes; first responder to bacterial infection |
| platelet | small cell fragment that starts blood clotting; no nucleus |

In [ ]:
USE_FULL_DATA = False    # True: all 17,092 images (35 MB download); the quiz numbers refer to the subset

SUBSET_URL = "https://raw.githubusercontent.com/ElofssonLab/kb8029-book/main/notebooks/data/bloodmnist-subset.npz"
FULL_URL = "https://zenodo.org/records/10519652/files/bloodmnist.npz?download=1"
CLASS_NAMES = ["basophil", "eosinophil", "erythroblast", "immature granulocyte",
               "lymphocyte", "monocyte", "neutrophil", "platelet"]

def download(url, filename):
    """Download url into the temporary directory once, and return the local path."""
    path = os.path.join(tempfile.gettempdir(), filename)
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path + ".part")
        os.replace(path + ".part", path)
    return path

def load_bloodmnist():
    """Return {"train"|"val"|"test": (images uint8 (N, 28, 28, 3), labels (N,))}."""
    if USE_FULL_DATA:
        d = np.load(download(FULL_URL, "bloodmnist.npz"))
    else:
        local = "data/bloodmnist-subset.npz"
        d = np.load(local if os.path.exists(local) else download(SUBSET_URL, "bloodmnist-subset.npz"))
    return {s: (d[f"{s}_images"], d[f"{s}_labels"].reshape(-1).astype(np.int64)) for s in ("train", "val", "test")}

data = load_bloodmnist()
X_train_img, y_train = data["train"]
X_val_img, y_val = data["val"]
X_test_img, y_test = data["test"]
print("training images:", X_train_img.shape, " validation:", X_val_img.shape, " test:", X_test_img.shape)
print("pixel value range:", X_train_img.min(), "-", X_train_img.max(), " (8-bit red, green, blue)")

fig, axes = plt.subplots(1, 8, figsize=(13, 2.2))
for k, ax in enumerate(axes):
    ax.imshow(X_train_img[np.where(y_train == k)[0][0]])
    ax.set_title(CLASS_NAMES[k].replace(" ", "\n"), fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

Count the classes. Is this training set balanced?

In [ ]:
class_counts = todo('count how many training images there are of each class (hint: np.bincount)')
for k, n in enumerate(class_counts):
    print(f"{k}  {CLASS_NAMES[k]:22s} {n:5d} images  ({n / len(y_train):.1%})")
most_common = todo('the class index with the most training images')
print("most abundant class:", CLASS_NAMES[most_common], "  least abundant:", CLASS_NAMES[int(np.argmin(class_counts))])

### A baseline before any model

A "classifier" that ignores the image and always answers the most common
class gets some images right for free. Its accuracy is the bar any real
model has to clear, and it shows why accuracy alone can mislead when
the classes are imbalanced.

In [ ]:
baseline_accuracy = todo('accuracy on the VALIDATION set of always predicting most_common')
print(f"always '{CLASS_NAMES[most_common]}': validation accuracy {baseline_accuracy:.3f}")

## 2. Training, validation and test sets

The three sets are BloodMNIST's own: the dataset's authors split the
images once, and everyone who reports results on it uses the same split,
so numbers from different papers can be compared. We train on the
training set, use the **validation set** to look at the model and make
choices, and keep the **test set** untouched until Section 8.

In [ ]:
print(f"training {len(y_train)}   validation {len(y_val)}   test {len(y_test)}")
print("validation class counts:", np.bincount(y_val, minlength=8))

## 3. Normalization, without leaking the test set

A common strategy is to **standardize**: subtract the mean and divide by
the standard deviation, so that inputs are centred on zero with unit
spread. A colour image has three channels (red, green, blue) with
different brightness, so we use one mean and one standard deviation
**per channel**. Session 8's rule: preprocessing statistics come from the
**training set only**, and are then applied unchanged to validation and
test data. Computing them on all the data is **leakage through
preprocessing**, one of the three routes from the Session 8 page;
Section 9 measures the second, **leakage through the samples**
(homologs), and using the test set only once (Section 8) guards against
the third, **leakage through the evaluation** (repeatedly inspecting the
test set while choosing models).

The images are arrays of shape (N, 28, 28, 3): image, row, column,
channel. Averaging over axes 0, 1 and 2 leaves one number per channel.

In [ ]:
X_train_raw = X_train_img / 255.0
X_val_raw = X_val_img / 255.0
X_test_raw = X_test_img / 255.0

pixel_mean = todo('the mean of each colour channel, over the TRAINING images only (3 numbers)')
pixel_std = todo('the standard deviation of each colour channel, TRAINING images only (3 numbers)')
print("training-set channel means (R, G, B):", np.round(pixel_mean, 4))
print("training-set channel stds  (R, G, B):", np.round(pixel_std, 4))

standardize = lambda X: ((X - pixel_mean) / pixel_std).astype(np.float32)
X_train, X_val, X_test = standardize(X_train_raw), standardize(X_val_raw), standardize(X_test_raw)
print(f"after standardizing: train mean {X_train.mean():+.4f}, std {X_train.std():.4f}")
print(f"                     val   mean {X_val.mean():+.4f}, std {X_val.std():.4f}   (close to, not exactly, 0 and 1)")

## 4. The model, as an `nn.Module`

Write a class with these layers, in order:

1. `nn.Flatten()`: a 28×28×3 image becomes a vector of 2,352 numbers;
2. `nn.Linear(2352, 10)` followed by `nn.Tanh()`: a hidden layer of 10 units;
3. `nn.Linear(10, 8)`: 8 outputs, one **logit** per cell type.

There is no softmax layer: as on the Session 8 page, the loss used inside
`train_model` (`nn.CrossEntropyLoss`) combines log-softmax with the
negative log-likelihood in one numerically stable operation, so the model
outputs raw logits.

In [ ]:
class BloodCellClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = todo('nn.Sequential of Flatten, Linear(2352, 10), Tanh, Linear(10, 8)')

    def forward(self, x):
        return self.net(x)

model = BloodCellClassifier()
print(model)
n_params = todo('total number of trainable parameters (sum of p.numel())')
print("trainable parameters:", n_params)

## 5. Training (a black box until Session 9)

`train_model` runs **stochastic gradient descent** in mini-batches of 128
images for 60 epochs, and records training and validation accuracy after
every epoch. You don't need to change it.

In [ ]:
def train_model(model, X_train, y_train, X_val, y_val, epochs=60, lr=0.01, batch_size=128, seed=SEED):
    '''Train `model` with mini-batch SGD and cross-entropy loss (Session 9 explains
    every line). Returns a dict of per-epoch training/validation accuracy.'''
    torch.manual_seed(seed)
    Xt, yt = torch.tensor(X_train), torch.tensor(y_train)
    Xv, yv = torch.tensor(X_val), torch.tensor(y_val)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    history = {"train_acc": [], "val_acc": []}
    for epoch in range(epochs):
        model.train()
        for batch in torch.randperm(len(Xt)).split(batch_size):
            optimizer.zero_grad()
            loss_fn(model(Xt[batch]), yt[batch]).backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            history["train_acc"].append((model(Xt).argmax(1) == yt).float().mean().item())
            history["val_acc"].append((model(Xv).argmax(1) == yv).float().mean().item())
    return history

torch.manual_seed(SEED)
model = BloodCellClassifier()
history = train_model(model, X_train, y_train, X_val, y_val)
print(f"after {len(history['train_acc'])} epochs: training accuracy {history['train_acc'][-1]:.3f}, "
      f"validation accuracy {history['val_acc'][-1]:.3f}")

plt.plot(history["train_acc"], label="training")
plt.plot(history["val_acc"], label="validation")
plt.axhline(baseline_accuracy, ls="--", color="grey", label="always the most common class")
plt.xlabel("epoch"); plt.ylabel("accuracy"); plt.ylim(0, 1); plt.legend(); plt.show()

## 6. From outputs to classes, and the confusion matrix

The model outputs 8 logits per image. `torch.softmax` turns them into
8 positive numbers that sum to 1 (estimated probabilities); the
predicted class is the one with the highest value.

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(torch.tensor(X_val))
probs = todo('softmax over the 8 outputs (dim=1), as a NumPy array')
predicted = todo('the predicted class for every validation image')
print("first image's probabilities:", np.round(probs[0], 3), " sum =", round(float(probs[0].sum()), 4))
print("first 10 predictions:", predicted[:10], "  true labels:", y_val[:10])

val_accuracy = todo('fraction of validation images predicted correctly')
print(f"validation accuracy: {val_accuracy:.3f}   (always '{CLASS_NAMES[most_common]}': {baseline_accuracy:.3f})")

In [ ]:
cm = confusion_matrix(y_val, predicted, labels=range(8))
fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(cm, cmap="Blues")
for i in range(8):
    for j in range(8):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=9,
                color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xticks(range(8)); ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
ax.set_yticks(range(8)); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel("predicted"); ax.set_ylabel("true")
ax.set_title("validation confusion matrix")
plt.tight_layout(); plt.show()

recall_per_class = cm.diagonal() / cm.sum(axis=1)
for k in np.argsort(recall_per_class):
    print(f"{CLASS_NAMES[k]:22s} recall {recall_per_class[k]:.2f}")

Look at the off-diagonal cells: which mistakes are common, and which
never happen? Read them against the table in Section 1. Immature
granulocytes are, mostly, neutrophils that have not finished maturing, so
the two form a continuum rather than two clean boxes; monocytes are
large cells with a nucleus of similar shape. A platelet, on the other
hand, is a small fragment with no nucleus and looks like nothing else.

### Immature granulocytes against the rest

An increase of immature granulocytes in the blood is a clinical warning
sign (for example of a severe infection), so finding them matters. Treat
"immature granulocyte" (class 3) as the positive class and every other
cell type as negative:

- predicted IG, truly IG: **true positive (TP)**
- predicted IG, truly not IG: **false positive (FP)**
- predicted not IG, truly IG: **false negative (FN)**
- predicted not IG, truly not IG: **true negative (TN)**

In [ ]:
POSITIVE = 3   # immature granulocyte
is_pos, pred_pos = (y_val == POSITIVE), (predicted == POSITIVE)
TP = todo('count of true positives')
FP = todo('count of false positives')
FN = todo('count of false negatives')
TN = todo('count of true negatives')
print(f"TP={TP}  FP={FP}  FN={FN}  TN={TN}   (total {TP+FP+FN+TN})")

precision = todo('precision for immature granulocytes')
recall = todo('recall for immature granulocytes')
f1 = todo('F1 for immature granulocytes')
mcc = matthews_corrcoef(is_pos, pred_pos)
print(f"immature granulocyte: precision {precision:.3f}  recall {recall:.3f}  F1 {f1:.3f}  MCC {mcc:.3f}")
print(f"accuracy of the IG-vs-rest call: {(TP + TN) / len(y_val):.3f}")

## 7. ROC curve and AUROC for immature granulocytes

The ROC curve needs a *score*, not a yes/no call: use the model's
softmax output for class 3.

In [ ]:
score_pos = todo('the softmax output for class 3 (immature granulocyte) for every validation image')
fpr, tpr, _ = roc_curve(is_pos, score_pos)
auroc = roc_auc_score(is_pos, score_pos)
plt.plot(fpr, tpr, label=f"immature granulocyte vs rest (AUROC {auroc:.3f})")
plt.plot([0, 1], [0, 1], "--", color="grey", label="random")
plt.xlabel("false-positive rate"); plt.ylabel("true-positive rate"); plt.legend(); plt.show()
print(f"AUROC, immature granulocyte vs rest: {auroc:.4f}")

## 8. How much data does the model need? Overfitting

Retrain the same model on smaller training sets, drawn from the 4,000
training images with the classes in the same proportions (a *stratified*
sample). The validation set stays the same 1,000 images every time.
Watch the gap between training and validation accuracy.

In [ ]:
sizes = [100, 1000, len(y_train)]
results = {}
for n in sizes:
    if n < len(y_train):
        tr_idx, _ = train_test_split(np.arange(len(y_train)), train_size=n, stratify=y_train, random_state=SEED)
    else:
        tr_idx = np.arange(len(y_train))
    torch.manual_seed(SEED)
    m = BloodCellClassifier()
    h = train_model(m, X_train[tr_idx], y_train[tr_idx], X_val, y_val)
    results[n] = (h["train_acc"][-1], h["val_acc"][-1])
    print(f"{n:5d} training images: training acc {results[n][0]:.3f}   validation acc {results[n][1]:.3f}"
          f"   gap {results[n][0] - results[n][1]:+.3f}")

plt.plot(sizes, [results[n][0] for n in sizes], "o-", label="training")
plt.plot(sizes, [results[n][1] for n in sizes], "o-", label="validation")
plt.xscale("log"); plt.xlabel("training images"); plt.ylabel("accuracy after 60 epochs")
plt.legend(); plt.show()

### The test set, used once

Validation has guided our choices (e.g. the training-set size). Now,
**once**, measure the model trained on all 4,000 images in Section 5 on
the 2,000 test images:

In [ ]:
with torch.no_grad():
    test_pred = model(torch.tensor(X_test)).argmax(1).numpy()
test_accuracy = todo('accuracy on the test images')
print(f"test accuracy: {test_accuracy:.3f}   (validation accuracy was {val_accuracy:.3f})")

## 9. Back to proteins: how leaky is a random split?

Session 8's localization dataset: 700 human proteins in 5 compartments,
clustered with MMseqs2 at 30% identity (precomputed, so no MMseqs2 is
needed). A **random** split ignores the clusters. How many validation
proteins end up with a homolog, i.e. a member of the same cluster, in
the training set? This needs only the accessions and labels (fetched
live from UniProt, ~10 s), no model. This is the same question as the
normalization in Section 3, from a different angle: in both cases
something about the validation data is already present in what the model
was trained on.

In [ ]:
import requests
from sklearn.model_selection import train_test_split

UNIPROT_CLASSES = {"Cytoplasm": "SL-0086", "Nucleus": "SL-0191", "Mitochondrion": "SL-0173",
                   "Secreted": "SL-0243", "Cell membrane": "SL-0039"}
CLASS_NAMES = list(UNIPROT_CLASSES)

def fetch_uniprot(sl_code, size=500):
    query = f"organism_id:9606 AND reviewed:true AND cc_scl_term:{sl_code} AND length:[50 TO 500]"
    r = requests.get("https://rest.uniprot.org/uniprotkb/search",
                     params={"query": query, "fields": "accession,sequence,cc_subcellular_location",
                             "format": "tsv", "size": size}, timeout=60)
    r.raise_for_status()
    return [tuple(line.split("\t")) for line in r.text.strip().split("\n")[1:] if len(line.split("\t")) == 3]

def is_unambiguous_single_location(location_text, target):
    location_text = re.sub(r"Note=.*", "", location_text)
    location_text = re.sub(r"\{[^}]*\}", "", location_text)
    if "Isoform" in location_text:
        return False
    terms = set()
    for statement in [s.strip() for s in location_text.split(".") if s.strip()]:
        body = statement.split(":", 1)[-1] if ":" in statement else statement
        top_term = re.split(r"[,;]", body)[0].strip()
        if top_term:
            terms.add(top_term)
    return len(terms) == 1 and next(iter(terms)) == target

entries = {c: [(a, s) for a, s, loc in fetch_uniprot(code) if is_unambiguous_single_location(loc, c)]
           for c, code in UNIPROT_CLASSES.items()}
n_per_class = min(len(v) for v in entries.values())
rng = np.random.RandomState(0)
accs, labels = [], []
for k, c in enumerate(CLASS_NAMES):
    for i in rng.choice(len(entries[c]), n_per_class, replace=False):
        accs.append(entries[c][i][0]); labels.append(k)
labels = np.array(labels)

TSV_URL = "https://raw.githubusercontent.com/ElofssonLab/kb8029-book/main/notebooks/data/session08-subcell-mmseqs-30.tsv"
TSV_LOCAL = "data/session08-subcell-mmseqs-30.tsv"
text = open(TSV_LOCAL).read() if os.path.exists(TSV_LOCAL) else requests.get(TSV_URL, timeout=30).text
member_to_cluster = {m: rep for rep, m in (line.split("\t") for line in text.strip().split("\n")[1:])}
clusters = np.array([member_to_cluster.get(a, a) for a in accs])
print(len(accs), "proteins in", len(set(clusters)), "clusters")

In [ ]:
idx = np.arange(len(labels))
tr, tmp = train_test_split(idx, test_size=0.30, stratify=labels, random_state=0)
va, te = train_test_split(tmp, test_size=0.5, stratify=labels[tmp], random_state=0)

train_clusters = set(clusters[tr])
has_homolog = todo('for each validation protein, is its cluster also in the training set?')
leak_fraction = todo('the fraction of validation proteins with a homolog in training')
print(f"random split (random_state=0): {has_homolog.sum()} of {len(va)} validation proteins "
      f"({leak_fraction:.1%}) have a >=30%-identity homolog in the training set")

**Last step:** save your notebook with all outputs (File → Download →
.ipynb) and upload it with the lab quiz on Canvas. If one of your numbers
falls outside the quiz's accepted range, the notebook lets us see whether
that is ordinary run-to-run variation in training.